<a href="https://colab.research.google.com/github/Abhinav9895/Generative-AI-Internship/blob/main/Day7/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")

print("Path to dataset files:",path)

100%|██████████| 1.99M/1.99M [00:01<00:00, 1.82MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/jp797498e/twitter-entity-sentiment-analysis/versions/2


In [2]:
import pandas as pd
import os

# List the contents of the downloaded dataset directory to find the correct file name
print("Files in dataset directory:")
for file_name in os.listdir(path):
    print(file_name)

# Using 'twitter_training.csv' as the data file based on the listed files.
# If you intend to use 'twitter_validation.csv', please change the filename below.
data_file_path = os.path.join(path, 'twitter_training.csv') # Concatenate path and filename

# Check if the file exists before attempting to read it
if not os.path.exists(data_file_path):
    raise FileNotFoundError(f"The file {data_file_path} does not exist. Please update the filename.")

df = pd.read_csv(data_file_path)

# Export to CSV without the pandas index column
df.to_csv('predictions.csv', index=False)

Files in dataset directory:
twitter_validation.csv
twitter_training.csv


### Data Preparation: Filtering Sentiments and Splitting Data

First, I will filter the dataset to include only the 'Positive', 'Negative', and 'Neutral' sentiments, as explicitly requested. Then, I'll split the data into training and testing sets, ensuring a balanced distribution of sentiments in both sets.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import re
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

# Rename columns for clarity (assuming df still has original column names)
df.columns = ['tweet_id', 'entity', 'sentiment', 'tweet_text']

# Drop rows where 'tweet_text' is missing
df.dropna(subset=['tweet_text'], inplace=True)

# Filter out 'Irrelevant' sentiment to focus on Positive, Negative, and Neutral
df_filtered = df[df['sentiment'].isin(['Positive', 'Negative', 'Neutral'])].copy()

# Define features (X) and target (y)
X = df_filtered['tweet_text']
y = df_filtered['sentiment']

# Split data into training and testing sets
# stratify=y ensures that the proportion of target labels is the same in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples (Positive, Negative, Neutral only): {len(X_train)}")
print(f"Testing samples (Positive, Negative, Neutral only): {len(X_test)}")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


Training samples (Positive, Negative, Neutral only): 48896
Testing samples (Positive, Negative, Neutral only): 12224


### Feature Engineering: TF-IDF and Enhanced VADER Sentiment Scores

I will use TF-IDF to convert the text data into numerical features. Additionally, to improve the model's understanding of negation, I will augment VADER sentiment scores with a custom negation handling function. These combined features will then be used to train the SVM model.

In [4]:
# Initialize TF-IDF Vectorizer (max_features can be tuned)
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit and transform the training data, then transform the test data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Initialize the VADER sentiment intensity analyzer
sia = SentimentIntensityAnalyzer()

def get_sentiment_scores(text):
    # Simple negation handling for VADER
    words = text.lower().split()
    negated_words = ['not', 'no', 'never', 'n\'t']
    modified_text = []
    for i, word in enumerate(words):
        if word in negated_words and i + 1 < len(words):
            # Mark the next word as negated
            modified_text.append(word)
            modified_text.append('NOT_' + words[i+1])
            words[i+1] = '' # Clear the original word to prevent re-processing
        elif word != '': # Only append if not already processed as a negated word
            modified_text.append(word)
    processed_text = ' '.join(modified_text)

    scores = sia.polarity_scores(processed_text)
    return scores['neg'], scores['neu'], scores['pos'], scores['compound']

# Apply the sentiment analysis to the training and test sets
X_train_sentiment = X_train.apply(lambda text: pd.Series(get_sentiment_scores(text), index=['neg', 'neu', 'pos', 'compound']))
X_test_sentiment = X_test.apply(lambda text: pd.Series(get_sentiment_scores(text), index=['neg', 'neu', 'pos', 'compound']))

# Convert TF-IDF sparse matrix to dense array for concatenation
X_train_tfidf_dense = X_train_tfidf.toarray()
X_test_tfidf_dense = X_test_tfidf.toarray()

# Concatenate TF-IDF features with VADER sentiment scores
X_train_combined = np.hstack((X_train_tfidf_dense, X_train_sentiment))
X_test_combined = np.hstack((X_test_tfidf_dense, X_test_sentiment))

print(f"Shape of X_train_combined: {X_train_combined.shape}")
print(f"Shape of X_test_combined: {X_test_combined.shape}")

Shape of X_train_combined: (48896, 5004)
Shape of X_test_combined: (12224, 5004)


### Model Training: Linear SVM and Prediction Function

Now, I will train a Linear Support Vector Machine (SVM) model using the combined TF-IDF and VADER features. After training, I will evaluate its performance and create a function that allows you to input a sentence and get its predicted sentiment.

In [5]:
from sklearn.svm import LinearSVC

# Train a LinearSVC model with the combined features
svm_model_enhanced = LinearSVC(random_state=42, max_iter=2000) # Increased max_iter for convergence
svm_model_enhanced.fit(X_train_combined, y_train)

# Make predictions on the test set with the enhanced SVM model
y_pred_svm_enhanced = svm_model_enhanced.predict(X_test_combined)

# Evaluate the enhanced SVM model
accuracy_svm_enhanced = accuracy_score(y_test, y_pred_svm_enhanced)
print(f"\nEnhanced SVM Model Accuracy: {accuracy_svm_enhanced:.4f}")
print("\nEnhanced SVM Classification Report:")
print(classification_report(y_test, y_pred_svm_enhanced))

# Define the prediction function
def predict_sentiment(sentence):
    # Preprocess the input sentence using the trained TF-IDF vectorizer
    sentence_tfidf = tfidf_vectorizer.transform([sentence])

    # Get sentiment scores for the sentence using the custom negation handling function
    neg, neu, pos, compound = get_sentiment_scores(sentence)
    sentiment_features = np.array([neg, neu, pos, compound]).reshape(1, -1)

    # Concatenate TF-IDF features with sentiment scores
    sentence_combined = np.hstack((sentence_tfidf.toarray(), sentiment_features))

    # Predict the sentiment using the enhanced SVM model
    sentiment = svm_model_enhanced.predict(sentence_combined)[0]

    return sentiment

# Test sentences for demonstration
example_sentence_1 = "I love this game so much! It's fantastic and very engaging."
example_sentence_2 = "This product is terrible. I regret buying it."
example_sentence_3 = "The weather is neither good nor bad today."

print("\n--- Testing the predict_sentiment function ---")
predicted_sentiment_1 = predict_sentiment(example_sentence_1)
print(f"Sentence: '{example_sentence_1}'\nPredicted Sentiment: {predicted_sentiment_1}\n")

predicted_sentiment_2 = predict_sentiment(example_sentence_2)
print(f"Sentence: '{example_sentence_2}'\nPredicted Sentiment: {predicted_sentiment_2}\n")

predicted_sentiment_3 = predict_sentiment(example_sentence_3)
print(f"Sentence: '{example_sentence_3}'\nPredicted Sentiment: {predicted_sentiment_3}\n")


Enhanced SVM Model Accuracy: 0.7929

Enhanced SVM Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.83      0.82      4472
     Neutral       0.78      0.72      0.75      3621
    Positive       0.79      0.81      0.80      4131

    accuracy                           0.79     12224
   macro avg       0.79      0.79      0.79     12224
weighted avg       0.79      0.79      0.79     12224


--- Testing the predict_sentiment function ---
Sentence: 'I love this game so much! It's fantastic and very engaging.'
Predicted Sentiment: Positive

Sentence: 'This product is terrible. I regret buying it.'
Predicted Sentiment: Negative

Sentence: 'The weather is neither good nor bad today.'
Predicted Sentiment: Positive



In [7]:
# Example: Predict the sentiment of a new sentence
my_new_sentence = "This movie was terrible!"
predicted_sentiment = predict_sentiment(my_new_sentence)
print(f"Sentence: '{my_new_sentence}'\nPredicted Sentiment: {predicted_sentiment}")

my_new_sentence_2 = "I have no strong feelings about this situation."
predicted_sentiment_2 = predict_sentiment(my_new_sentence_2)
print(f"Sentence: '{my_new_sentence_2}'\nPredicted Sentiment: {predicted_sentiment_2}")

Sentence: 'This movie was terrible!'
Predicted Sentiment: Negative
Sentence: 'I have no strong feelings about this situation.'
Predicted Sentiment: Neutral
